In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV,StratifiedKFold
import pickle

In [ ]:
df = pd.read_csv('data/input.csv')

In [ ]:
df.head()

## preproceesing:

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(12, 8))  # Adjust the figure size
sns.heatmap(df.corr(),annot=True,cmap='coolwarm',square=True,linewidths=0.5)        

## Classifying diabetes

In [ ]:
df.sample()

In [ ]:
x = df.iloc[:,0:-1]
y = df.iloc[:,-1]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
X_train  = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

##  Logistic Regression :

In [ ]:
# Before hyperparameter tuning
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
accuracy_lr_before = accuracy_score(y_test, y_pred_lr)

# Hyperparameter tuning
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(LogisticRegression(), param_grid_lr, cv=skf, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train, y_train)

# After hyperparameter tuning
best_lr = grid_lr.best_estimator_
y_pred_lr_tuned = best_lr.predict(X_test)
accuracy_lr_after = accuracy_score(y_test, y_pred_lr_tuned)

print("Logistic Regression Accuracy")
print("Before tuning:", accuracy_lr_before)
print("After tuning:", accuracy_lr_after)


## 2. K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Before hyperparameter tuning
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
accuracy_knn_before = accuracy_score(y_test, y_pred_knn)

# Hyperparameter tuning
param_grid_knn = {'n_neighbors': [3, 5, 7, 9, 11]}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=skf, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train, y_train)

# After hyperparameter tuning
best_knn = grid_knn.best_estimator_
y_pred_knn_tuned = best_knn.predict(X_test)
accuracy_knn_after = accuracy_score(y_test, y_pred_knn_tuned)

print("K-Nearest Neighbors Accuracy")
print("Before tuning:", accuracy_knn_before)
print("After tuning:", accuracy_knn_after)

## 3. Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

# Before hyperparameter tuning
svm = SVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
accuracy_svm_before = accuracy_score(y_test, y_pred_svm)

# Hyperparameter tuning
param_grid_svm = {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1]}
grid_svm = GridSearchCV(SVC(), param_grid_svm, cv=skf, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_train, y_train)

# After hyperparameter tuning
best_svm = grid_svm.best_estimator_
y_pred_svm_tuned = best_svm.predict(X_test)
accuracy_svm_after = accuracy_score(y_test, y_pred_svm_tuned)

print("Support Vector Machine Accuracy")
print("Before tuning:", accuracy_svm_before)
print("After tuning:", accuracy_svm_after)


## 4. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Before hyperparameter tuning
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
accuracy_dt_before = accuracy_score(y_test, y_pred_dt)

# Hyperparameter tuning
param_grid_dt = {'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 10, 20]}
grid_dt = GridSearchCV(DecisionTreeClassifier(), param_grid_dt, cv=skf, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_train, y_train)

# After hyperparameter tuning
best_dt = grid_dt.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_test)
accuracy_dt_after = accuracy_score(y_test, y_pred_dt_tuned)

print("Decision Tree Accuracy")
print("Before tuning:", accuracy_dt_before)
print("After tuning:", accuracy_dt_after)


## 5. Random Forest (Ensemble)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Before hyperparameter tuning
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
accuracy_rf_before = accuracy_score(y_test, y_pred_rf)

# Expanded Hyperparameter tuning
param_dist_rf = {
    'n_estimators': [50, 100, 200, 300, 400],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

random_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist_rf,
    n_iter=100,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
random_rf.fit(X_train, y_train)

# After hyperparameter tuning
best_rf = random_rf.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test)
accuracy_rf_after = accuracy_score(y_test, y_pred_rf_tuned)

print("Random Forest Accuracy")
print("Before tuning:", accuracy_rf_before)
print("After tuning:", accuracy_rf_after)

## Gradient boosting (ensemble)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Before hyperparameter tuning
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
accuracy_gb_before = accuracy_score(y_test, y_pred_gb)

# Hyperparameter tuning
param_grid_gb = {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 7]}
grid_gb = GridSearchCV(GradientBoostingClassifier(), param_grid_gb, cv=skf, scoring='accuracy', n_jobs=-1)
grid_gb.fit(X_train, y_train)

# After hyperparameter tuning
best_gb = grid_gb.best_estimator_
y_pred_gb_tuned = best_gb.predict(X_test)
accuracy_gb_after = accuracy_score(y_test, y_pred_gb_tuned)

print("Gradient Boosting Accuracy")
print("Before tuning:", accuracy_gb_before)
print("After tuning:", accuracy_gb_after)


## 7. AdaBoost (Ensemble)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Before hyperparameter tuning
ab = AdaBoostClassifier(random_state=42)
ab.fit(X_train, y_train)
y_pred_ab = ab.predict(X_test)
accuracy_ab_before = accuracy_score(y_test, y_pred_ab)

# Expanded hyperparameter tuning
param_grid_ab = {
    'n_estimators': [50, 100, 200, 300, 500],
    'learning_rate': [0.01, 0.1, 0.5, 1.0, 1.5, 2.0],
    'algorithm': ['SAMME', 'SAMME.R']
}

# Using StratifiedKFold for better cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_ab = GridSearchCV(AdaBoostClassifier(random_state=42), param_grid_ab, cv=skf, scoring='accuracy', n_jobs=-1)
grid_ab.fit(X_train, y_train)

# After hyperparameter tuning
best_ab = grid_ab.best_estimator_
y_pred_ab_tuned = best_ab.predict(X_test)
accuracy_ab_after = accuracy_score(y_test, y_pred_ab_tuned)

print("AdaBoost Accuracy")
print("Before tuning:", accuracy_ab_before)
print("After tuning:", accuracy_ab_after)

## 8. Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Before hyperparameter tuning
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
accuracy_nb_before = accuracy_score(y_test, y_pred_nb)

# Note: GaussianNB does not have hyperparameters to tune with GridSearchCV in this context.

print("Naive Bayes Accuracy")
print("Before tuning:", accuracy_nb_before)
# Same accuracy after "tuning" since there's no hyperparameter to tune.
print("After tuning:", accuracy_nb_before)


## 9. Linear Discriminant Analysis (LDA)

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Before hyperparameter tuning
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
y_pred_lda = lda.predict(X_test)
accuracy_lda_before = accuracy_score(y_test, y_pred_lda)

# Hyperparameter tuning
param_grid_lda = {'solver': ['svd', 'lsqr', 'eigen']}
grid_lda = GridSearchCV(LinearDiscriminantAnalysis(), param_grid_lda, cv=skf, scoring='accuracy', n_jobs=-1)
grid_lda.fit(X_train, y_train)

# After hyperparameter tuning
best_lda = grid_lda.best_estimator_
y_pred_lda_tuned = best_lda.predict(X_test)
accuracy_lda_after = accuracy_score(y_test, y_pred_lda_tuned)

print("Linear Discriminant Analysis Accuracy")
print("Before tuning:", accuracy_lda_before)
print("After tuning:", accuracy_lda_after)

## 10. XGBoost (Ensemble)

In [ ]:
###import xgboost as xgb
#from sklearn.metrics import accuracy_score
#from sklearn.model_selection import GridSearchCV

# Before hyperparameter tuning
#xg = xgb.XGBClassifier(random_state=42)
#xg.fit(X_train, y_train)
#y_pred_xg = xg.predict(X_test)
#accuracy_xg_before = accuracy_score(y_test, y_pred_xg)

# Hyperparameter tuning
#param_grid_xg = {
#    'n_estimators': [50, 100, 200],
#    'learning_rate': [0.01, 0.1, 0.2],
#    'max_depth': [3, 5, 7],
#    'subsample': [0.8, 0.9, 1.0],
#    'colsample_bytree': [0.8, 0.9, 1.0]
#}

#grid_xg = GridSearchCV(xgb.XGBClassifier(random_state=42), param_grid_xg, cv=skf, scoring='accuracy', n_jobs=-1)
#grid_xg.fit(X_train, y_train)

# After hyperparameter tuning
#best_xg = grid_xg.best_estimator_
#y_pred_xg_tuned = best_xg.predict(X_test)
#accuracy_xg_after = accuracy_score(y_test, y_pred_xg_tuned)

#print("XGBoost Accuracy")
#print("Before tuning:", accuracy_xg_before)
#print("After tuning:", accuracy_xg_after)


### Ada boost is giving best performance

In [ ]:
ab = AdaBoostClassifier(random_state=42)
ab.fit(X_train, y_train)
y_pred_ab = ab.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_ab)
print(accuracy)

# Obtain cross-validation scores
cv_scores = cross_val_score(ab, X_train, y_train, cv=5, scoring='accuracy')

# Print the cross-validation scores
print("Cross-Validation Scores:", cv_scores)

# Calculate and print the mean cross-validation score
print("Mean Cross-Validation Score:", cv_scores.mean())

Based on the provided accuracy score (0.7727272727272727) and the cross-validation scores, the model seems to be performing reasonably well. Here's how you can interpret the results:

- The accuracy score on the test set is 0.7727, which indicates that the model correctly predicts the class label for approximately 77.27% of the instances in the test set.

- The cross-validation scores range from approximately 0.6992 to 0.7787 across different folds, with a mean cross-validation score of approximately 0.7509.

- The mean cross-validation score is close to the accuracy score on the test set, indicating that the model's performance is consistent across different folds of the data.


In [ ]:
with open ('diabetes_predictor(adaboost).pkl','wb') as file:
    pickle.dump(ab,file)